# 제조 ASR 일반화 최종 검증 — v1·v2 회귀평가와 독립 Test v3

신규 일반화 개발 데이터에서 Validation 1위로 선택된 large-v3 5시간 LoRA를 고정한 뒤, 디코딩·후처리·양자화를 Validation에서만 선택합니다. 기존 Test v1과 실패 이력이 있는 v2는 회복 여부를 보는 회귀평가로 사용하고, 완전히 분리된 신규 Test v3만 독립 확인평가로 사용합니다.

> 모든 음성은 AI 합성 데이터입니다. 결과는 합성 음향 프로필 일반화 근거이며 실제 작업자·공장 성능이나 생산 준비 완료를 증명하지 않습니다.

## 0. 실행 승인 스위치

기본값은 비용 작업을 막기 위해 False입니다. 이번 승인 실행에서는 두 값을 True로 변경합니다. 완료 결과는 불변 경로에서 재사용되어 같은 Test를 반복 추론하지 않습니다.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/Pronesis9758/aias-specialist-asr.git'
GITHUB_BRANCH = 'codex/whisper-benchmark-quantization'
PROJECT_DIR = '/content/AIAS'
DRIVE_ROOT = '/content/drive/MyDrive/AI_Specialist_ASR_Project'

# 신규 Test v3 WAV가 아직 없을 때만 True로 실행합니다.
GENERATE_TEST_V3_AUDIO = False
# 병합·Validation 선택·v1/v2/v3 평가·통합 집계를 실행합니다.
RUN_FINAL_GENERALIZATION = False

## 1. A100·고용량 RAM·Google Drive 확인

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

drive_root = Path('/content/drive/MyDrive')
if not drive_root.is_dir():
    drive.mount('/content/drive')
if not drive_root.is_dir():
    raise RuntimeError('Google Drive 연결을 확인할 수 없습니다.')
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임이 필요합니다.')
gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in gpu_name.upper() or gpu_memory < 35:
    raise RuntimeError(f'A100 40GB 이상이 필요합니다: {gpu_name} {gpu_memory:.1f}GB')
print('GPU:', gpu_name, f'{gpu_memory:.1f}GB')
!free -h

## 2. 최신 코드 동기화와 기존 의존성 복원

In [ ]:
if not Path(PROJECT_DIR).exists():
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', GITHUB_REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'merge', '--ff-only', f'origin/{GITHUB_BRANCH}'], check=True)
os.chdir(PROJECT_DIR)
print('Git commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Python:', sys.version)

In [ ]:
# 저장소에 선언된 train extra만 설치하며 별도의 임의 패키지 조합은 만들지 않습니다.
%pip uninstall -y torchao gradio gradio-client
%pip install -q -e '.[train]' 'transformers>=4.46,<5' 'peft>=0.14,<0.19' 'edge-tts>=7,<8'

def run_aias(*args):
    command = [sys.executable, '-m', 'aias_specialist.cli', *args]
    print('\nRunning:', ' '.join(command), flush=True)
    subprocess.run(command, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})

## 3. 기존 데이터·pilot 결과와 Test v3 계획 확인

In [ ]:
import json
import pandas as pd
import yaml

GENERALIZATION_CONFIG = 'configs/synthetic_manufacturing_generalization_v3.yaml'
LORA_DIR = Path(DRIVE_ROOT) / 'artifacts/training/synthetic-manufacturing-lora-a100-generalization-v3'
LORA_SELECTION = LORA_DIR / 'lora_selection.yaml'
DEV_MANIFEST = Path(DRIVE_ROOT) / 'data/synthetic/manufacturing_generalization_v3/manifest.csv'
V1_MANIFEST = Path(DRIVE_ROOT) / 'data/synthetic/manufacturing_7200/manifest.csv'
V2_MANIFEST = Path(DRIVE_ROOT) / 'data/synthetic/manufacturing_test_v2/manifest.csv'
TEST_V3_SPEC = 'configs/data/synthetic_manufacturing_test_v3.yaml'
TEST_V3_ROOT = Path(DRIVE_ROOT) / 'data/synthetic/manufacturing_test_v3'
TEST_V3_MANIFEST = TEST_V3_ROOT / 'manifest.csv'

required = [LORA_SELECTION, DEV_MANIFEST, V1_MANIFEST, V2_MANIFEST]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'필수 기존 산출물이 없습니다: {missing}')
curve = pd.read_csv(LORA_DIR / 'lora_learning_curve.csv')
pilot = curve.loc[curve['stage_id'].eq('pilot-5h')].sort_values('rank')
display(pilot[['model_id', 'train_samples', 'domain_term_recall', 'cer', 'wer', 'rank']])
selection_payload = yaml.safe_load(LORA_SELECTION.read_text(encoding='utf-8'))
if selection_payload['model_id'] != 'large-v3':
    raise RuntimeError(f"pilot-5h 고정 선택이 large-v3가 아닙니다: {selection_payload['model_id']}")
run_aias('plan-synthetic-dataset', '--spec', TEST_V3_SPEC)
test_v3_plan = pd.read_csv(TEST_V3_ROOT / 'manifest.plan.csv')
display(test_v3_plan.groupby('difficulty_group').size().rename('samples'))
display(json.loads((TEST_V3_ROOT / 'plan_summary.json').read_text(encoding='utf-8')))

## 4. 신규 독립 Test v3 합성 및 교차 무결성 검사

In [ ]:
if GENERATE_TEST_V3_AUDIO:
    run_aias('synthesize-dataset', '--spec', TEST_V3_SPEC, '--concurrency', '4')
else:
    print('Test v3 음성 생성을 건너뜁니다. 최초 실행에서만 True로 변경하세요.')
if RUN_FINAL_GENERALIZATION and not TEST_V3_MANIFEST.exists():
    raise FileNotFoundError('Test v3 최종 manifest가 없습니다. 먼저 음성을 생성하세요.')
if TEST_V3_MANIFEST.exists():
    from aias_specialist.confirmatory import validate_confirmatory_cohort
    audit = validate_confirmatory_cohort(
        V1_MANIFEST, TEST_V3_MANIFEST,
        additional_reference_manifest_paths=[V2_MANIFEST, DEV_MANIFEST],
        expected_samples=600, minimum_speakers=30,
        minimum_term_occurrences=1000, minimum_negative_samples=100,
    )
    print(json.dumps(audit, ensure_ascii=False, indent=2))

## 5. large-v3 5시간 LoRA 병합과 Validation 디코딩 선택

Test를 열기 전에 새 Validation 1,200건만 사용하여 beam·hotwords·VAD를 선택합니다.

In [ ]:
if RUN_FINAL_GENERALIZATION:
    run_aias('merge-selected-lora', '--selection', str(LORA_SELECTION), '--config', GENERALIZATION_CONFIG)
    run_aias('decoding-sweep-selected-lora', '--selection', str(LORA_SELECTION), '--config', GENERALIZATION_CONFIG)
lora_payload = yaml.safe_load(LORA_SELECTION.read_text(encoding='utf-8'))
DECODING_ID = f"synthetic-lora-decoding-{lora_payload['selection_id']}"
DECODING_DIR = Path(DRIVE_ROOT) / 'artifacts/benchmarks' / DECODING_ID
DECODING_SELECTION = DECODING_DIR / 'model_selection.yaml'
if DECODING_SELECTION.exists():
    display(pd.read_csv(DECODING_DIR / 'benchmark_comparison.csv'))

## 6. Validation IR·NN 안전성 탐색

In [ ]:
CORRECTION_ID = f"generalization-v3-correction-{lora_payload['selection_id']}"
CORRECTION_DIR = Path(DRIVE_ROOT) / 'artifacts/correction' / CORRECTION_ID
CORRECTION_SELECTION = CORRECTION_DIR / 'correction_selection.yaml'
if RUN_FINAL_GENERALIZATION:
    correction_spec = yaml.safe_load(Path('configs/correction/synthetic_manufacturing_generalization_v3.yaml').read_text(encoding='utf-8'))
    correction_spec['correction_sweep']['id'] = CORRECTION_ID
    runtime_correction = Path('/content/aias_generalization_correction.yaml')
    runtime_correction.write_text(yaml.safe_dump(correction_spec, allow_unicode=True, sort_keys=False), encoding='utf-8')
    run_aias('correction-sweep', '--spec', str(runtime_correction), '--selection', str(DECODING_SELECTION))
    if not CORRECTION_SELECTION.exists():
        run_aias('select-correction', '--sweep-dir', str(CORRECTION_DIR), '--reviewer', 'AUTOMATED_SYNTHETIC_VALIDATION', '--reason', 'Validation 품질 및 문장 악화율 gate 기준 선택', '--automated-proxy')
if CORRECTION_SELECTION.exists():
    display(pd.read_csv(CORRECTION_DIR / 'correction_sweep_comparison.csv'))

## 7. Validation FP16·INT8 비교와 정확도 우선 선택

In [ ]:
QUANT_ID = f"generalization-v3-quantization-{lora_payload['selection_id']}"
QUANT_DIR = Path(DRIVE_ROOT) / 'artifacts/quantization' / QUANT_ID
QUANT_SELECTION = QUANT_DIR / 'quantization_selection.yaml'
RUNTIME_FINAL_CONFIG = Path('/content/aias_generalization_final_config.yaml')
if RUN_FINAL_GENERALIZATION:
    final_config = yaml.safe_load(Path(GENERALIZATION_CONFIG).read_text(encoding='utf-8'))
    correction_choice = yaml.safe_load(CORRECTION_SELECTION.read_text(encoding='utf-8'))['selection']['correction']
    final_config['correction'] = correction_choice
    RUNTIME_FINAL_CONFIG.write_text(yaml.safe_dump(final_config, allow_unicode=True, sort_keys=False), encoding='utf-8')
    quant_spec = yaml.safe_load(Path('configs/quantization/synthetic_manufacturing_generalization_v3.yaml').read_text(encoding='utf-8'))
    quant_spec['quantization']['id'] = QUANT_ID
    quant_spec['quantization']['base_config'] = str(RUNTIME_FINAL_CONFIG)
    runtime_quant = Path('/content/aias_generalization_quantization.yaml')
    runtime_quant.write_text(yaml.safe_dump(quant_spec, allow_unicode=True, sort_keys=False), encoding='utf-8')
    run_aias('quantization-sweep', '--spec', str(runtime_quant), '--selection', str(DECODING_SELECTION))
    quant_table = pd.read_csv(QUANT_DIR / 'quantization_comparison.csv')
    display(quant_table)
    completed = quant_table.loc[quant_table['status'].eq('completed')].copy()
    passing = completed.loc[completed['quality_target_pass'].astype(bool)]
    if passing.empty:
        raise RuntimeError('품질 gate를 통과한 양자화 후보가 없습니다.')
    accuracy_choice = passing.sort_values(['domain_term_recall', 'cer', 'wer'], ascending=[False, True, True]).iloc[0]
    if not QUANT_SELECTION.exists():
        run_aias('select-quantization', '--quantization-dir', str(QUANT_DIR), '--variant-id', str(accuracy_choice['member_id']), '--reviewer', 'AUTOMATED_SYNTHETIC_VALIDATION', '--reason', 'Validation Recall 우선, CER/WER 순위로 고정', '--automated-proxy')

## 8. 설정 고정 후 Test v1·v2·v3 평가

v1과 v2는 회귀평가이며, v2가 v3 개발 데이터 설계에 영향을 주었다는 사실을 명시합니다. Test v3는 세 기존 manifest와 문장·화자·음성 hash·음향 프로필이 겹치지 않는 독립 확인평가입니다.

In [ ]:
V1_RUNTIME_CONFIG = Path('/content/aias_generalization_v1_eval.yaml')
V1_RESULT = QUANT_DIR / 'final_test_result.json'
V2_RESULT = QUANT_DIR / 'confirmatory/generalization-regression-v2/confirmatory_test_result.json'
V3_RESULT = QUANT_DIR / 'confirmatory/independent-heldout-v3/confirmatory_test_result.json'
if RUN_FINAL_GENERALIZATION:
    v1_config = yaml.safe_load(Path('configs/evaluation/synthetic_manufacturing_generalization_v1.yaml').read_text(encoding='utf-8'))
    v1_config['correction'] = yaml.safe_load(CORRECTION_SELECTION.read_text(encoding='utf-8'))['selection']['correction']
    V1_RUNTIME_CONFIG.write_text(yaml.safe_dump(v1_config, allow_unicode=True, sort_keys=False), encoding='utf-8')
    run_aias('finalize-evaluation', '--selection', str(QUANT_SELECTION), '--config', str(V1_RUNTIME_CONFIG))
    run_aias('confirmatory-evaluation', '--selection', str(QUANT_SELECTION), '--config', str(V1_RUNTIME_CONFIG), '--manifest', str(V2_MANIFEST), '--cohort-id', 'generalization-regression-v2', '--cohort-role', 'regression', '--development-influence', '--expected-samples', '600', '--reference-result', str(V1_RESULT), '--additional-reference-manifest', str(DEV_MANIFEST))
    run_aias('confirmatory-evaluation', '--selection', str(QUANT_SELECTION), '--config', str(V1_RUNTIME_CONFIG), '--manifest', str(TEST_V3_MANIFEST), '--cohort-id', 'independent-heldout-v3', '--cohort-role', 'confirmatory', '--expected-samples', '600', '--reference-result', str(V1_RESULT), '--additional-reference-manifest', str(V2_MANIFEST), '--additional-reference-manifest', str(DEV_MANIFEST))

## 9. 통합 지표·95% 신뢰구간·v2 전후 paired 비교

In [ ]:
GENERALIZATION_RESULT_DIR = Path(DRIVE_ROOT) / 'artifacts/generalization/final-large-v3-pilot5h'
historical_candidates = sorted(Path(DRIVE_ROOT).glob('artifacts/quantization/*/confirmatory/speaker-heldout-v2/confirmatory_test_result.json'))
historical_v2 = None
for candidate in historical_candidates:
    payload = json.loads(candidate.read_text(encoding='utf-8'))
    metrics = payload.get('metrics', {}).get('test_v2', {})
    if abs(float(metrics.get('domain_term_recall', -1)) - 0.8233) < 0.01:
        historical_v2 = candidate
        break
if RUN_FINAL_GENERALIZATION:
    command = ['generalization-summary', '--v1-result', str(V1_RESULT), '--v2-result', str(V2_RESULT), '--v3-result', str(V3_RESULT), '--config', str(V1_RUNTIME_CONFIG), '--output-dir', str(GENERALIZATION_RESULT_DIR), '--bootstrap-resamples', '1000']
    if historical_v2 is not None:
        command.extend(['--historical-v2-result', str(historical_v2)])
    run_aias(*command)
comparison_path = GENERALIZATION_RESULT_DIR / 'generalization_comparison.csv'
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
    display(pd.read_csv(GENERALIZATION_RESULT_DIR / 'generalization_confidence_intervals.csv'))
    before_after = GENERALIZATION_RESULT_DIR / 'test_v2_before_after.csv'
    if before_after.exists():
        display(pd.read_csv(before_after))

## 10. 연구 결론 판정

독립 Test v3가 Recall≥85%, CER≤7%, WER≤15%를 모두 통과해야 합성 음향 조건 일반화가 확인된 것으로 판정합니다. 실제 현장 적용은 별도 승인된 shadow 평가와 사람 검수를 거쳐야 합니다.

In [ ]:
if comparison_path.exists():
    results = pd.read_csv(comparison_path)
    independent = results.loc[results['cohort'].eq('test_v3')].iloc[0]
    print('INDEPENDENT_TEST_V3_GATE:', 'PASS' if independent['quality_gate_pass'] else 'FAIL')
    print('Recall:', f"{independent['domain_term_recall']:.2%}")
    print('Precision:', f"{independent['domain_term_precision']:.2%}")
    print('F1:', f"{independent['domain_term_f1']:.2%}")
    print('CER:', f"{independent['cer']:.2%}")
    print('WER:', f"{independent['wer']:.2%}")